# 07 - Freddie Mac Feature Engineering and Leakage Review

## Milestone objective

Convert the validated 2015-2023 loan-level modeling dataset into a stable, auditable predictor dataset for Logistic Regression and XGBoost. This notebook explains why each source field is selected, transformed, retained only as metadata, or excluded.

The governing rule is simple: **a predictor must be known at origination and must be representable consistently when a future cohort arrives.**

## How features are chosen

Feature selection is not based on p-values alone. A p-value answers a narrow conditional-association question and can be misleading with hundreds of thousands of observations, nonlinear effects, correlated predictors, or temporal drift. We therefore use five gates:

1. **Timing:** Was the value known at loan origination?
2. **Domain relevance:** Is there a defensible relationship to borrower, collateral, or loan-contract risk?
3. **Temporal availability:** Is the field available throughout the 2015-2017 development window and usable later?
4. **Stability and transferability:** Will the field generalize across years and, later, to Fannie Mae?
5. **Redundancy and leakage:** Does it duplicate another field, act like an identifier, or contain post-origination information?

Statistical diagnostics will still be used later to understand relationships and model behavior, but they will not replace these design checks.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.freddie_features import (
    CATEGORICAL_FEATURES,
    LABEL_COLUMNS,
    NUMERIC_FEATURES,
    PREDICTOR_COLUMNS,
    apply_preprocessing_contract,
    build_engineered_features,
    feature_decision_table,
    fit_preprocessing_contract,
    validate_feature_dataset,
)

## Step 1 - Load the validated modeling dataset

Notebook 06 established one eligible row per loan, reconciled origination records to the 24-month target, and retained the event-window fields for auditing. We begin from that validated artifact rather than rebuilding labels here.

In [2]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'Freddie_Mac'
INPUT_PATH = DATA_DIR / 'freddie_modeling_dataset_2015_2023_24m.parquet'

modeling_data = pd.read_parquet(INPUT_PATH)
print('Input path:', INPUT_PATH)
print('Shape:', modeling_data.shape)
print('Years:', sorted(modeling_data['SOURCE YEAR'].unique().tolist()))
print('Duplicate loan/year keys:', modeling_data.duplicated(['SOURCE YEAR', 'LOAN IDENTIFIER']).sum())

Input path: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m.parquet
Shape: (437668, 59)
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Duplicate loan/year keys: 0


## Step 2 - Review every origination field

The decision table is the audit trail for feature selection. It covers all 31 Freddie Mac origination fields. Exclusion does not mean a field is universally useless; it means the field is unsuitable for this project's stable, origination-time core model for the stated reason.

In [3]:
decisions = feature_decision_table()
display(decisions)
display(decisions['decision'].value_counts().rename_axis('decision').to_frame('field_count'))

,source_field,decision,final_feature,reason
0,CLASSIC FICO,selected,FICO_SCORE,Core borrower credit-risk measure; use the cle...
1,FIRST PAYMENT DATE,excluded,,Calendar timing is represented by cohort metad...
2,FIRST TIME HOMEBUYER INDICATOR,selected,FIRST_TIME_HOMEBUYER,Origination-time borrower characteristic with ...
3,MATURITY DATE,excluded,,Redundant with first-payment timing and origin...
4,METROPOLITAN STATISTICAL AREA (MSA) OR METROPO...,excluded,,High-cardinality geography with changing defin...
5,MORTGAGE INSURANCE PERCENTAGE (MI %),selected,MI_PERCENT; HAS_MORTGAGE_INSURANCE,Captures mortgage-insurance protection and hig...
6,NUMBER OF UNITS,selected,NUMBER_OF_UNITS,Property structure is known at origination and...
7,OCCUPANCY STATUS,selected,OCCUPANCY_STATUS,"Primary residence, second home, and investment..."
8,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),selected-derived,CLTV_MINUS_LTV; HAS_SUBORDINATE_FINANCING,Use the incremental CLTV-LTV gap instead of ra...
9,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,selected,ORIGINAL_DTI,Core borrower affordability measure; use the d...


,field_count
decision,
selected,18
excluded,9
selected-derived,1
selected-transformed,1
metadata-only,1
selected-harmonized,1


## Step 3 - Enforce the leakage boundary

Loan identifiers and cohort fields remain available for reconciliation, temporal splitting, and reporting, but they are not predictors. Event dates, observation-window fields, and the 24-month horizon are post-origination target-audit information and are removed entirely from the feature table. This prevents the model from receiving information that would not exist when a new loan is scored.

In [4]:
engineered = build_engineered_features(modeling_data)

print('Engineered shape:', engineered.shape)
print('Predictors:', len(PREDICTOR_COLUMNS))
print('Numeric predictors:', len(NUMERIC_FEATURES))
print('Categorical predictors:', len(CATEGORICAL_FEATURES))
display(engineered.head())

Engineered shape: (437668, 37)
Predictors: 29
Numeric predictors: 15
Categorical predictors: 14


,SOURCE YEAR,LOAN IDENTIFIER,ORIGINATION YEAR,ORIGINATION QUARTER,ORIGINATION COHORT,COHORT ROLE,FICO_SCORE,MI_PERCENT,ORIGINAL_DTI,ORIGINAL_LTV,...,PROPERTY_TYPE,LOAN_PURPOSE,ORIGINAL_LOAN_TERM,NUMBER_OF_BORROWERS_GROUP,SUPER_CONFORMING_FLAG,SPECIAL_ELIGIBILITY_PROGRAM,HARP_INDICATOR,INTEREST_ONLY_INDICATOR,TARGET_90_PLUS,TARGET_COMPOSITE_CREDIT_EVENT
0,2015,F15Q10000025,2015,Q1,2015Q1,DEVELOPMENT_2015_2017,734.0,0,15.0,65.0,...,SF,P,180,2+,N,<NA>,N,N,0,0
1,2015,F15Q10000031,2015,Q1,2015Q1,DEVELOPMENT_2015_2017,797.0,0,26.0,37.0,...,PU,P,360,2+,N,<NA>,N,N,0,0
2,2015,F15Q10000084,2015,Q1,2015Q1,DEVELOPMENT_2015_2017,670.0,12,33.0,90.0,...,SF,N,180,2+,N,<NA>,N,N,0,0
3,2015,F15Q10000087,2015,Q1,2015Q1,DEVELOPMENT_2015_2017,731.0,0,NaN,73.0,...,SF,N,360,2+,N,<NA>,Y,N,0,0
4,2015,F15Q10000157,2015,Q1,2015Q1,DEVELOPMENT_2015_2017,668.0,0,30.0,75.0,...,SF,N,180,2+,N,<NA>,N,N,0,0


## Step 4 - Create limited, domain-defensible features

The pipeline keeps engineering deliberately conservative:

- `LOG_ORIGINAL_UPB` reduces balance-scale skew without changing loan ordering.
- `CLTV_MINUS_LTV` measures the additional leverage attributable to subordinate financing. Raw CLTV is not also retained because LTV, CLTV, and their difference would be algebraically redundant for Logistic Regression.
- Mortgage-insurance, subordinate-financing, and LTV-above-80 indicators represent documented mortgage-domain thresholds or structures.
- Missingness flags preserve the possibility that non-disclosure itself carries information.

We do not create broad score or DTI bands at this stage. Keeping continuous information supports both models; bins can later be tested as an interpretable sensitivity analysis without replacing the core values.

In [5]:
derived_columns = [
    'LOG_ORIGINAL_UPB',
    'CLTV_MINUS_LTV',
    'HAS_MORTGAGE_INSURANCE',
    'HAS_SUBORDINATE_FINANCING',
    'HIGH_LTV_GT_80',
]
display(engineered[derived_columns].describe().T)

missing_before_contract = (
    engineered[PREDICTOR_COLUMNS].isna().mean().mul(100).sort_values(ascending=False)
)
display(missing_before_contract[missing_before_contract.gt(0)].to_frame('missing_percent'))

,count,mean,std,min,25%,50%,75%,max
LOG_ORIGINAL_UPB,437668.0,12.324898,0.572175,9.21044,11.951187,12.367345,12.7484,14.414348
CLTV_MINUS_LTV,437666.0,0.347802,2.918841,0.00000,0.000000,0.000000,0.0000,809.000000
HAS_MORTGAGE_INSURANCE,437668.0,0.287423,0.452561,0.00000,0.000000,0.000000,1.0000,1.000000
HAS_SUBORDINATE_FINANCING,437668.0,0.026623,0.160979,0.00000,0.000000,0.000000,0.0000,1.000000
HIGH_LTV_GT_80,437668.0,0.292429,0.454879,0.00000,0.000000,0.000000,1.0000,1.000000


,missing_percent
SPECIAL_ELIGIBILITY_PROGRAM,93.719669
ORIGINAL_DTI,1.968387
FICO_SCORE,0.015308
CLTV_MINUS_LTV,0.000457
ORIGINAL_LTV,0.000457


## Step 5 - Fit preprocessing rules on 2015-2017 only

Medians and retained category levels are learned only from the development cohorts. Later cohorts must not influence preprocessing because that would allow future population information to leak backward into model development.

Numerical missing values receive development-period medians after explicit missingness indicators are created. Categorical missingness becomes `MISSING`; rare development categories and categories first observed in later years become `OTHER`. This gives all cohorts the same schema and makes previously unseen levels safe at scoring time. One-hot encoding is intentionally deferred to the model pipeline, where it will also be fitted on development data only.

In [6]:
contract = fit_preprocessing_contract(
    engineered,
    minimum_category_count=100,
)
features = apply_preprocessing_contract(engineered, contract)

print('Contract training years:', contract['training_years'])
print('Minimum retained-category count:', contract['minimum_category_count'])
display(pd.Series(contract['numeric_medians'], name='training_median').to_frame())
display(pd.Series({key: len(value) for key, value in contract['category_levels'].items()}, name='retained_levels').to_frame())
print('Missing predictor cells after contract:', int(features[PREDICTOR_COLUMNS].isna().sum().sum()))

Contract training years: [2015, 2016, 2017]
Minimum retained-category count: 100


,training_median
FICO_SCORE,756.000000
MI_PERCENT,0.000000
ORIGINAL_DTI,36.000000
ORIGINAL_LTV,79.000000
ORIGINAL_INTEREST_RATE,4.000000
LOG_ORIGINAL_UPB,12.235636
CLTV_MINUS_LTV,0.000000
HAS_MORTGAGE_INSURANCE,0.000000
HAS_SUBORDINATE_FINANCING,0.000000
HIGH_LTV_GT_80,0.000000


,retained_levels
FIRST_TIME_HOMEBUYER,2
NUMBER_OF_UNITS,4
OCCUPANCY_STATUS,3
CHANNEL,3
AMORTIZATION_TYPE,1
PROPERTY_STATE,51
PROPERTY_TYPE,5
LOAN_PURPOSE,3
ORIGINAL_LOAN_TERM,5
NUMBER_OF_BORROWERS_GROUP,2


Missing predictor cells after contract: 0


## Step 6 - Preserve the temporal experiment

The feature table assigns 2015-2017 to model development and 2018-2023 to sequential out-of-time evaluation. `SOURCE YEAR`, cohort, and role remain metadata rather than predictors. This is essential: the research question concerns whether a frozen model remains reliable as populations change, so the model must not be taught the answer by receiving calendar year as an input.

The target distribution is summarized but not resampled here. SMOTE, undersampling, or class weights would change the probability distribution and can damage calibration. Any later imbalance treatment must be evaluated inside the modeling design, not during dataset construction.

In [7]:
role_summary = (
    features.groupby(['SOURCE YEAR', 'COHORT ROLE'], as_index=False)
    .agg(
        loans=('LOAN IDENTIFIER', 'size'),
        events=('TARGET_COMPOSITE_CREDIT_EVENT', 'sum'),
        event_rate=('TARGET_COMPOSITE_CREDIT_EVENT', 'mean'),
    )
)
role_summary['event_rate_percent'] = role_summary['event_rate'] * 100
display(role_summary.round(3))

,SOURCE YEAR,COHORT ROLE,loans,events,event_rate,event_rate_percent
0,2015,DEVELOPMENT_2015_2017,47244,248,0.005,0.525
1,2016,DEVELOPMENT_2015_2017,48076,324,0.007,0.674
2,2017,DEVELOPMENT_2015_2017,48278,391,0.008,0.810
3,2018,EVALUATION_2018_2023,47417,1222,0.026,2.577
4,2019,EVALUATION_2018_2023,48611,2227,0.046,4.581
5,2020,EVALUATION_2018_2023,49665,802,0.016,1.615
6,2021,EVALUATION_2018_2023,49490,451,0.009,0.911
7,2022,EVALUATION_2018_2023,49386,903,0.018,1.828
8,2023,EVALUATION_2018_2023,49501,876,0.018,1.770


## Step 7 - Validate and save the feature contract

The final checks require unique and complete loan/year keys, valid binary labels, no missing or infinite predictor values, both temporal roles, and complete exclusion of post-origination audit fields. The saved Parquet file is read back and validated again so serialization problems cannot pass silently.

In [8]:
validation = validate_feature_dataset(features)
for key, value in validation.items():
    print(f'{key}: {value}')

OUTPUT_PATHS = {
    'features': DATA_DIR / 'freddie_features_2015_2023_24m.parquet',
    'decisions': DATA_DIR / 'freddie_feature_decisions.csv',
    'contract': DATA_DIR / 'freddie_feature_contract.json',
    'validation': DATA_DIR / 'freddie_feature_validation.json',
}

features.to_parquet(OUTPUT_PATHS['features'], index=False, engine='pyarrow')
decisions.to_csv(OUTPUT_PATHS['decisions'], index=False)
with OUTPUT_PATHS['contract'].open('w', encoding='utf-8') as file:
    json.dump(contract, file, indent=2)
with OUTPUT_PATHS['validation'].open('w', encoding='utf-8') as file:
    json.dump(validation, file, indent=2)

saved_features = pd.read_parquet(OUTPUT_PATHS['features'])
saved_validation = validate_feature_dataset(saved_features)
assert validation == saved_validation

print('Saved and read-back validated:')
for name, path in OUTPUT_PATHS.items():
    print(f'  {name}: {path}')

rows: 437668
columns: 37
predictor_count: 29
numeric_predictor_count: 15
categorical_predictor_count: 14
missing_required_columns: []
forbidden_post_origination_columns: []
duplicate_loan_year_keys: 0
missing_loan_year_keys: 0
missing_predictor_values_after_contract: 0
missing_labels: 0
invalid_label_values: {}
all_numeric_predictors_finite: True
observed_cohort_roles: ['DEVELOPMENT_2015_2017', 'EVALUATION_2018_2023']
Saved and read-back validated:
  features: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_features_2015_2023_24m.parquet
  decisions: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_feature_decisions.csv
  contract: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_feature_contract.json
  validation: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_feature_validation.json


## Milestone conclusion

The project now has a documented, reproducible feature boundary rather than an informal list of columns. All predictors are available at origination; post-origination target evidence is excluded; definition changes are either harmonized or rejected; preprocessing is frozen from 2015-2017; and the same feature contract can be applied to every 2018-2023 cohort.

The next research milestone is the peer-reviewed literature review on population drift, performance degradation, early-warning signals, and retraining benefit. After the benchmark methodology is grounded in that literature, the same frozen feature table will support Logistic Regression and XGBoost training.